In [1]:
import pandas as pd
import numpy as np
import gc

from sklearn.model_selection import train_test_split
import torch
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer, create_model_and_transforms
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from transformers import AutoModel, AutoTokenizer
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import plotly as px
from plotly import graph_objects as go
from sklearn.preprocessing import StandardScaler
from peft import LoraConfig, get_peft_model
from torch.amp import autocast, GradScaler

In [2]:
pd.options.display.max_columns = None
sample_dataset = pd.read_csv("./chexpert_sample_dataset.csv", index_col=0)
sample_dataset

,path_to_image,deid_patient_id,report,sub_report,target,report_length,sub_report_length
0,train/patient18877/study1/view1_frontal.jpg,patient18877,NARRATIVE:\nCHEST: Two views.\nCOMPARISON: 12-...,\nThere is plate-like atelectasis seen at the ...,0,110,76
1,train/patient28169/study1/view1_frontal.jpg,patient28169,IMPRESSION:\n \nCARDIAC SILHOUETTE IS AT THE U...,\n \nCARDIAC SILHOUETTE IS AT THE UPPER LIMITS...,1,95,74
2,train/patient58769/study1/view1_frontal.jpg,patient58769,NARRATIVE:\nChest 1 View: 1-11-2020\n \nHISTOR...,\n \nUnchanged right internal jugular venous ...,1,86,48
3,train/patient54916/study1/view1_frontal.jpg,patient54916,NARRATIVE:\nChest 1 View: 6/6/2008\n \nHISTORY...,\n \n1. PORTABLE UPRIGHT FRONTAL CHEST RADIO...,1,100,68
4,train/patient01990/study1/view1_frontal.jpg,patient01990,NARRATIVE:\nCHEST:\nCLINICAL HISTORY: Check li...,\nThere is an endotracheal tube with its tip a...,1,152,118
...,...,...,...,...,...,...,...
21995,train/patient08833/study1/view1_frontal.jpg,patient08833,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\n \nRedemonstration of right IJ central venou...,0,109,57
21996,train/patient63772/study1/view1_frontal.jpg,patient63772,"NARRATIVE:\nChest 1 View Portable, 09-01-14\n ...",\n \n1.STABLE POSITIONING OF THE LEFT CHEST ...,0,121,75
21997,train/patient46004/study1/view1_frontal.jpg,patient46004,"NARRATIVE:\nEXAM: Chest Post Needle Biopsy, 20...",\n \n1. SEMI-UPRIGHT FRONTAL VIEW OF THE CHEST...,1,101,59
21998,train/patient07484/study1/view1_frontal.jpg,patient07484,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 9-28-20...,\n \n UPRIGHT PORTABLE CHEST WITHOUT COMPARIS...,1,64,35


In [3]:
X = sample_dataset.drop(columns=["target"])
y = sample_dataset["target"]

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [4]:
lora_X_train, lora_X_test, lora_y_train, lora_y_test = X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy()
lora_X_train = lora_X_train.reset_index(drop=True)
lora_X_test = lora_X_test.reset_index(drop=True)
lora_y_train = lora_y_train.reset_index(drop=True)
lora_y_test = lora_y_test.reset_index(drop=True)

# Fine tuning BioMedCLIP with LoRA (Low Rank Adaptation)

In [5]:
clip_lora_model, lora_preprocess_train, lora_preprocess_val = create_model_and_transforms('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
lora_tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

In [6]:
class LoraDatasetTrain(Dataset):
    def __init__(self, dataframe, labels):
        self.dataframe = dataframe
        self.labels = labels
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # Load raw data
        raw_image_path = "./CheXpert/CheXpert-v1.0-small/" + self.dataframe.loc[idx, "path_to_image"]
        raw_report_text = self.dataframe.loc[idx, "report"]
        label = self.labels[idx]
        
        # Transform raw pixels into a 3x224x224 tensor
        pil_img = Image.open(raw_image_path).convert("RGB")
        processed_image_tensor = lora_preprocess_train(pil_img) # Shape: [3, 224, 224]
        
        # Transform raw string into a 256-long token ID tensor
        tokenized_text_tensor = lora_tokenizer(raw_report_text, context_length=256).squeeze(0) # Shape: [256]
        
        return processed_image_tensor, tokenized_text_tensor, torch.tensor(label, dtype=torch.float32)

class LoraDatasetTest(Dataset):
    def __init__(self, dataframe, labels):
        self.dataframe = dataframe
        self.labels = labels
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # Load raw data
        raw_image_path = "./CheXpert/CheXpert-v1.0-small/" + self.dataframe.loc[idx, "path_to_image"]
        raw_report_text = self.dataframe.loc[idx, "report"]
        label = self.labels[idx]
        
        # Transform raw pixels into a 3x224x224 tensor
        pil_img = Image.open(raw_image_path).convert("RGB")
        processed_image_tensor = lora_preprocess_val(pil_img) # Shape: [3, 224, 224]
        
        # Transform raw string into a 256-long token ID tensor
        tokenized_text_tensor = lora_tokenizer(raw_report_text, context_length=256).squeeze(0) # Shape: [256]
        
        return processed_image_tensor, tokenized_text_tensor, torch.tensor(label, dtype=torch.float32)

In [7]:
lora_train_dataset = LoraDatasetTrain(lora_X_train, lora_y_train)
lora_test_dataset = LoraDatasetTest(lora_X_test, lora_y_test)

lora_train_loader = DataLoader(lora_train_dataset, batch_size=32, shuffle=True)
lora_val_loader = DataLoader(lora_test_dataset, batch_size=32, shuffle=False)

In [8]:
class LoraClassifier(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, dropout=0.3):
        super().__init__()
        
        self.cross_attention = nn.MultiheadAttention(
                    embed_dim=embed_dim, 
                    num_heads=num_heads, 
                    dropout=dropout,
                    batch_first=True
                )
        
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
    
        # Add a small Feed-Forward Network layer (FFN) inside the attention block 
        # This mirrors a true standard Transformer block and stabilizes representation warping
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim)
        )
    
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.LayerNorm(256), # BatchNorm stabilizes linear scaling transitions!
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        
    def forward(self, img_patches, text_tokens):
        norm_text = self.layer_norm1(text_tokens)
        # Cross-Attention Core
        attn_output, _ = self.cross_attention(
            query=norm_text, 
            key=img_patches, 
            value=img_patches
        )
        x = (attn_output + text_tokens)
        
        # FFN stabilization step
        x = self.layer_norm2(self.ffn(x)) + x
        
        # Step 3: Max-pooling across sequence dimension 
        # This captures the strongest text-visual features and ignores inactive padding tokens
        x_pooled, _ = torch.max(x, dim=1)
        # Step 4: Predict
        return self.classifier(x_pooled)

In [9]:
# --- [FULL PIPELINE WRAPPER WITH LORA] ---
class BioMedCLIPWithLoRA(nn.Module):
    def __init__(self, biomedclip_base_model, r=8, alpha=16):
        super().__init__()
        
        # Target the Attention projections of BioMedCLIP.
        lora_config = LoraConfig(
            r=r,
            lora_alpha=alpha,
            target_modules=["qkv", "query", "value"], 
            lora_dropout=0.1,
            bias="none"
        )
        
        # Wrap the base model. PEFT automatically freezes base weights 
        # and injects the trainable A and B matrices into the target_modules.
        self.biomedclip = get_peft_model(biomedclip_base_model, lora_config)
        
        # Instantiate custom fusion head
        self.fusion_head = LoraClassifier()
        
    def forward(self, image_tensors, token_id_tensors):
        # 1. Extract text tokens (Confirmed 3D: [Batch, 256, 768])
        text_outputs = self.biomedclip.base_model.model.text.transformer(token_id_tensors)
        text_tokens = text_outputs.last_hidden_state 
        
        # 2. Extract raw image tokens from the timm Vision Transformer
        # We call the trunk but bypass its global pooling block if it has one set
        img_patches = self.biomedclip.base_model.model.visual.trunk.forward_features(image_tensors)
        
        # --- THE FIX FOR THE 2D ASSERTION ---
        # If timm returned [64, 768], it means it globally pooled the patches.
        # To get the unpooled patch grid, we pull the features right before the head.
        if len(img_patches.shape) == 2:
            # Plan B: Extract from the patch embedding and transformer blocks directly
            # This completely bypasses any global pooling settings in the trunk wrapper
            x = self.biomedclip.base_model.model.visual.trunk.patch_embed(image_tensors)
            x = self.biomedclip.base_model.model.visual.trunk._pos_embed(x)
            x = self.biomedclip.base_model.model.visual.trunk.patch_drop(x)
            x = self.biomedclip.base_model.model.visual.trunk.norm_pre(x)
            img_patches = self.biomedclip.base_model.model.visual.trunk.blocks(x)
            img_patches = self.biomedclip.base_model.model.visual.trunk.norm(img_patches)
            
        # Quick sanity printout for your first step to confirm shapes in console (optional)
        # print(f"Text Shape: {text_tokens.shape}, Vision Shape: {img_patches.shape}")
        
        # Both are now guaranteed 3D tensors of shape:
        # Text: [Batch, 256, 768]
        # Vision: [Batch, 197, 768]
        return self.fusion_head(img_patches, text_tokens)

In [17]:
def lora_train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.

    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Binary Cross Entropy for classification).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).

    Returns:
    history: Dictionary containing loss and accuracy for both training and validation.
    """

    # Dictionary to store training & validation loss and accuracy over epochs
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    # Initialize GradScaler for Mixed Precision (handles FP16 scaling)
    scaler = GradScaler('cuda')

    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions

        # Training loop
        for inputs_img, inputs_txt, labels in train_loader:
            inputs_img = inputs_img.to(device, non_blocking=True)
            inputs_txt = inputs_txt.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                outputs = model(inputs_img, inputs_txt).squeeze(1)  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
            # Scale loss and backpropagate in mixed precision
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # loss.backward()  # Backpropagation (compute gradients)
            # optimizer.step()  # Update model parameters

            total_loss += loss.item()  # Accumulate batch loss
            correct += ((torch.sigmoid(outputs) >= 0.5).float() == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)

        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs_img, inputs_txt, labels in val_loader:
                inputs_img = inputs_img.to(device, non_blocking=True)
                inputs_txt = inputs_txt.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with autocast('cuda'):
                    outputs = model(inputs_img, inputs_txt).squeeze(1) # Forward pass
                    loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                val_correct += ((torch.sigmoid(outputs) >= 0.5).float() == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        scheduler.step()
        # Capture current learning rate for tracking
        current_lr = optimizer.param_groups[0]['lr']

        # Store metrics in history dictionary
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)

        # Monitor real VRAM footprint vs cached VRAM
        real_vram = torch.cuda.memory_allocated(device) / 1024**3
        cached_vram = torch.cuda.memory_reserved(device) / 1024**3

        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, LR: {current_lr:.6f}")
        print(f"   -> Real Math VRAM: {real_vram:.2f} GB | PyTorch Cache: {cached_vram:.2f} GB\n")

        torch.save({
            'epoch': epoch+1,  # Crucial for resuming exactly where you left off
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': history["loss"][-1], # Optional but helpful
            'val_loss': history["val_loss"][-1],
            'accuracy': history["accuracy"][-1], # Optional but helpful
            'val_accuracy': history["val_accuracy"][-1],
        }, "./biomed_lora_models/"+f"checkpoint_{epoch+1}.pth")

    return history  # Return training history


In [11]:
# Clear Python's main garbage collector
gc.collect()

# Clear the PyTorch VRAM cache
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Assuming 'base_biomed_model' is your loaded raw BioMedCLIP model instance
lora_model = BioMedCLIPWithLoRA(biomedclip_base_model=clip_lora_model, r=8, alpha=16).to(device)

# Verify what is actually training
# You'll notice only the LoRA matrices and your fusion head are True
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print(f"Trainable: {name}")

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(lora_model.parameters(), lr=1e-5, weight_decay=1e-2)

# Cosine Annealing Scheduler over 50 epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

Trainable: biomedclip.base_model.model.visual.trunk.blocks.0.attn.qkv.lora_A.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.0.attn.qkv.lora_B.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.1.attn.qkv.lora_A.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.1.attn.qkv.lora_B.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.2.attn.qkv.lora_A.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.2.attn.qkv.lora_B.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.3.attn.qkv.lora_A.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.3.attn.qkv.lora_B.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.4.attn.qkv.lora_A.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.4.attn.qkv.lora_B.default.weight
Trainable: biomedclip.base_model.model.visual.trunk.blocks.5.attn.qkv.lora_A.def

In [18]:
# Launch the training block
history_lora = lora_train(
    model=lora_model, 
    train_loader=lora_train_loader, 
    val_loader=lora_val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    scheduler = scheduler,
    epochs=30
)

Epoch [1/30], Loss: 0.3765, Acc: 0.8339, Val Loss: 0.4110, Val Acc: 0.8214, LR: 0.000009
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [2/30], Loss: 0.3538, Acc: 0.8453, Val Loss: 0.4066, Val Acc: 0.8145, LR: 0.000009
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [3/30], Loss: 0.3450, Acc: 0.8508, Val Loss: 0.3857, Val Acc: 0.8309, LR: 0.000009
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [4/30], Loss: 0.3331, Acc: 0.8539, Val Loss: 0.3821, Val Acc: 0.8311, LR: 0.000009
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [5/30], Loss: 0.3204, Acc: 0.8610, Val Loss: 0.3911, Val Acc: 0.8234, LR: 0.000008
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [6/30], Loss: 0.3161, Acc: 0.8655, Val Loss: 0.3889, Val Acc: 0.8286, LR: 0.000008
   -> Real Math VRAM: 0.88 GB | PyTorch Cache: 5.43 GB

Epoch [7/30], Loss: 0.3046, Acc: 0.8693, Val Loss: 0.3764, Val Acc: 0.8323, LR: 0.000008
   -> Real Math VRAM: 0.88 GB | PyTorch C

In [19]:
fig = go.Figure(data=[
    go.Scatter(y=history_lora["loss"], name="Training Loss", mode="lines"),
    go.Scatter(y=history_lora["val_loss"], name="Validation Loss", mode="lines")
])
fig.update_layout(title="Training and Validation Loss", xaxis_title="Epochs", yaxis_title="Loss")
fig.show()